# Practice: clusters and anomalies
*AI from scratch, stage 3* · lesson: https://ihor.sh/ai/clusters-and-anomalies.html

Distance at work: the stations nearest Grand Central, a kNN classifier and how to pick its k, k-means written by
hand and by scikit-learn, a Gaussian mixture's soft memberships, and an isolation forest flagging unusual subway
elevator outages. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)`
and `answer(n)` are there when you're stuck. On Kaggle, turn on *Internet* in the notebook settings.

Stations: © OpenStreetMap contributors (ODbL). Outages: this site's MTA archive.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['.sort_values("km") puts the smallest distances first; .head(5) keeps five rows.', 'closest = st.sort_values("km").head(5)'], 2: ['knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)', 'acc5 = knn.score(Xte, yte)'], 3: ['[something(k) for k in ks] makes one value per k.', 'aucs = [auc_for_k(k) for k in ks]'], 4: ['A list comprehension over j in range(k), turned into an array with np.array(...).', 'centers = np.array([X[labels == j].mean(axis=0) for j in range(k)])'], 5: ['km = KMeans(5, n_init=10, random_state=0).fit(X)', 'ari5 = adjusted_rand_score(st["railroad"], km.labels_)'], 6: ['Fit it like any scikit-learn model: GaussianMixture(...).fit(X).', 'gm = GaussianMixture(5, random_state=0, n_init=3).fit(X), then p = gm.predict_proba(X)'], 7: ['iso = IsolationForest(contamination=0.03, random_state=0).fit(F)', 'flagged = df[iso.predict(F) == -1] keeps the rows where the comparison is True.']}
_ANSWERS = {1: 'Y2xvc2VzdCA9IHN0LnNvcnRfdmFsdWVzKCJrbSIpLmhlYWQoNSk=', 2: 'a25uID0gS05laWdoYm9yc0NsYXNzaWZpZXIobl9uZWlnaGJvcnM9NSkuZml0KFh0ciwgeXRyKQphY2M1ID0ga25uLnNjb3JlKFh0ZSwgeXRlKQ==', 3: 'YXVjcyA9IFthdWNfZm9yX2soaykgZm9yIGsgaW4ga3Nd', 4: 'Y2VudGVycyA9IG5wLmFycmF5KFtYW2xhYmVscyA9PSBqXS5tZWFuKGF4aXM9MCkgZm9yIGogaW4gcmFuZ2UoayldKQ==', 5: 'a20gPSBLTWVhbnMoNSwgbl9pbml0PTEwLCByYW5kb21fc3RhdGU9MCkuZml0KFgpCmFyaTUgPSBhZGp1c3RlZF9yYW5kX3Njb3JlKHN0WyJyYWlscm9hZCJdLCBrbS5sYWJlbHNfKQ==', 6: 'Z20gPSBHYXVzc2lhbk1peHR1cmUoNSwgcmFuZG9tX3N0YXRlPTAsIG5faW5pdD0zKS5maXQoWCkKcCA9IGdtLnByZWRpY3RfcHJvYmEoWCk=', 7: 'aXNvID0gSXNvbGF0aW9uRm9yZXN0KGNvbnRhbWluYXRpb249MC4wMywgcmFuZG9tX3N0YXRlPTApLmZpdChGKQpmbGFnZ2VkID0gZGZbaXNvLnByZWRpY3QoRikgPT0gLTFd'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import io, json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
data = json.load(urllib.request.urlopen(DATA + "rail-stations.json"))
st = pd.DataFrame(data["rows"], columns=data["columns"])
st["x"] = (st["lon"] + 74.0) * 111.32 * np.cos(np.radians(40.75))     # km east of Manhattan
st["y"] = (st["lat"] - 40.75) * 110.57                                  # km north
X = st[["x", "y"]].values
colors = st["railroad"].map({"MNR": "tab:green", "LIRR": "tab:blue", "NJT": "tab:orange"})
plt.figure(figsize=(5, 5)); plt.scatter(st["x"], st["y"], c=colors, s=8); plt.gca().set_aspect("equal")
plt.xlabel("km east of Manhattan"); plt.ylabel("km north"); plt.show()

## 1. Distances
`km` is every station's distance from Grand Central (computed for you, with Pythagoras on the x, y kilometers). Set
`closest` to the 5 stations with the smallest `km`: sort the table by that column and take the first five rows.

In [ ]:
gc = st[st["name"] == "Grand Central Terminal"].iloc[0]
st["km"] = np.sqrt((st["x"] - gc["x"]) ** 2 + (st["y"] - gc["y"]) ** 2)
closest = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
closest[["name", "railroad", "km"]] if closest is not None else None

In [ ]:
check(1, lambda: closest["name"].iloc[0] == "Grand Central Terminal" and len(closest) == 5,
      "five rows, starting with Grand Central itself at 0 km")

## 2. Neighbors vote
Split the stations 75/25 (done for you), then fit `knn`, a `KNeighborsClassifier(n_neighbors=5)`, on the training
stations and set `acc5` to its accuracy on the test stations: `knn.score(X_test, y_test)` gives exactly that.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, st["railroad"], test_size=0.25, random_state=0, stratify=st["railroad"])
knn = acc5 = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(acc5)

In [ ]:
check(2, lambda: acc5 > 0.97, "99–100%: the railroads barely overlap on the map, so neighbors almost always share one")

## 3. Choosing k on harder data
The bank calls from the features lesson (numbers only, the leaking `duration` removed): `auc_for_k(k)` fits kNN
with k neighbors and returns its test AUC. Set `aucs` to the list of AUCs for every k in `ks`, with a list
comprehension.

In [ ]:
from sklearn.metrics import roc_auc_score
bank = pd.read_csv(DATA + "bank.csv")
yb = (bank["y"] == "yes").astype(int)
Xb = bank.select_dtypes("number").drop(columns="duration")
Xbtr, Xbte, ybtr, ybte = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)

def auc_for_k(k):
    model = KNeighborsClassifier(k).fit(Xbtr, ybtr)
    return roc_auc_score(ybte, model.predict_proba(Xbte)[:, 1])

ks = [1, 3, 5, 11, 25, 51, 101, 201]
aucs = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
if aucs:
    plt.semilogx(ks, aucs, "o-"); plt.xlabel("k"); plt.ylabel("test AUC"); plt.show()
print(aucs)

In [ ]:
check(3, lambda: aucs[0] < 0.65 and max(aucs) > 0.75,
      "k = 1 memorizes single noisy calls (AUC about 0.59); around 100 neighbors it reaches about 0.78")

## 4. k-means by hand
`kmeans(X, k)` alternates the lesson's two steps. The **assign** step is written: each point's nearest center. Write
the **move** step: `centers` becomes, for each cluster `j`, the mean of the points labeled `j`
(`X[labels == j].mean(axis=0)`), as one array.

In [ ]:
def kmeans(X, k, steps=100, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(steps):
        dist = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)   # (points, centers)
        labels = dist.argmin(axis=1)                                     # assign: the nearest center
        # YOUR CODE HERE (stuck? hint(4), then answer(4))
        ...
    return labels, centers

labels, centers = kmeans(X, 4)
my_inertia = ((X - centers[labels]) ** 2).sum()
print("inertia:", my_inertia)

In [ ]:
from sklearn.cluster import KMeans
best = KMeans(4, n_init=10, random_state=0).fit(X).inertia_
check(4, lambda: my_inertia < best * 1.05, "within a few percent of scikit-learn's best of 10 starts (about 426,500 km²); without the move step the centers never move")

## 5. Do clusters find the railroads?
Fit `km = KMeans(5, n_init=10, random_state=0)` on `X`, and set `ari5` to `adjusted_rand_score(st["railroad"],
km.labels_)`: 1 if the clusters matched the railroads exactly, about 0 for chance.

In [ ]:
from sklearn.metrics import adjusted_rand_score
km = ari5 = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
if km is not None:
    plt.figure(figsize=(5, 5)); plt.scatter(st["x"], st["y"], c=km.labels_, s=8, cmap="tab10"); plt.gca().set_aspect("equal"); plt.show()
print(ari5)

In [ ]:
check(5, lambda: 0.4 < ari5 < 0.6, "about 0.5: the clusters are groups of nearby stations, which only partly line up with who runs them")

## 6. Soft clusters
Fit `gm`, a `GaussianMixture(5, random_state=0, n_init=3)`, on `X`, and set `p` to `gm.predict_proba(X)`: one row
per station, one column per blob, each row adding up to 1.

In [ ]:
from sklearn.mixture import GaussianMixture
gm = p = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
if p is not None:
    unsure = np.argsort(p.max(axis=1))[:5]
    print(pd.DataFrame(p[unsure].round(2), index=st["name"].values[unsure]))

In [ ]:
check(6, lambda: p.shape == (len(st), 5) and np.allclose(p.sum(axis=1), 1),
      "a (396, 5) array of probabilities, each row adding up to 1")

## 7. Unusual outages
This site's archive of subway elevator and escalator outages, with the log of how long each lasted and of the MTA's
estimate in `F`. Fit `iso`, an `IsolationForest(contamination=0.03, random_state=0)` (it flags the 3% easiest to
isolate), on `F`, and set `flagged` to the rows of `df` it flags: `iso.predict(F)` is −1 for those.

In [ ]:
from sklearn.ensemble import IsolationForest
o = json.load(urllib.request.urlopen(DATA + "outages.json"))
df = pd.DataFrame(o["rows"], columns=o["columns"])
F = np.column_stack([np.log10(df["hours"]), np.log10(df["estimated"])])
iso = flagged = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
if flagged is not None:
    plt.scatter(F[:, 0], F[:, 1], s=8, alpha=0.5)
    plt.scatter(np.log10(flagged["hours"]), np.log10(flagged["estimated"]), s=80, facecolors="none", edgecolors="r")
    plt.xlabel("log10 actual hours"); plt.ylabel("log10 estimated hours"); plt.show()
flagged

In [ ]:
check(7, lambda: 711.13 in flagged["hours"].values and 8 <= len(flagged) <= 13,
      "about 11 outages (3% of 340), including the month-long 711-hour maintenance")

Look at what was flagged: month-long planned maintenance (rare but right), a 2.4-minute "outage" (likely a glitch),
repairs that took a small fraction of their estimate. The detector ranks; a person decides what each one means.

## Next
Back to the path: https://ihor.sh/ai/ · scikit-learn's clustering guide compares many methods side by side:
https://scikit-learn.org/stable/modules/clustering.html